<a href="https://colab.research.google.com/github/adtrmn/Road-to-be-a-Data-Analyst/blob/main/Day%202%20-%20Python%2C%20SQL%2C%20Excel%2C%20Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop Analisis Data End-to-End: Python, SQL & Excel Dashboard

Mini project: Mengambil data saham dari Yahoo Finance, membersihkan data, menganalisis dengan SQL (SQLite), lalu mengekspor hasil ke Excel untuk dibuat dashboard.

In [ ]:
# Install library (jalankan sekali)
!pip -q install yfinance openpyxl


In [ ]:
import pandas as pd #untuk manipulasi data
import yfinance as yf #akses ke url yahoo finance
import sqlite3
from IPython.display import display


## Step 1 - Scraping Data dari Yahoo Finance

In [ ]:
ticker = "BBCA.JK"
started = "2020-01-01"
ended = "2025-12-31"

df = yf.download(ticker, start=started, end=ended, auto_adjust=False)
df = df.reset_index()
display(df.head())
print(df.shape)


[*********************100%***********************]  1 of 1 completed


Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,BBCA.JK,BBCA.JK,BBCA.JK,BBCA.JK,BBCA.JK,BBCA.JK
0,2020-01-02,5540.172363,6690.0,6780.0,6680.0,6695.0,49445000
1,2020-01-03,5631.266113,6800.0,6800.0,6725.0,6750.0,47755500
2,2020-01-06,5577.437500,6735.0,6750.0,6690.0,6720.0,27300000
3,2020-01-07,5581.578613,6740.0,6770.0,6730.0,6735.0,45022500
4,2020-01-08,5531.890137,6680.0,6745.0,6670.0,6670.0,53692500


(1447, 7)


## Step 2 - Data Cleaning

In [ ]:
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df.columns = [str(c).strip() for c in df.columns]
df = df.dropna().copy()
df["Date"] = pd.to_datetime(df["Date"])
display(df.head())

,Date,Adj Close,Close,High,Low,Open,Volume
0,2020-01-02,5540.172363,6690.0,6780.0,6680.0,6695.0,49445000
1,2020-01-03,5631.266113,6800.0,6800.0,6725.0,6750.0,47755500
2,2020-01-06,5577.437500,6735.0,6750.0,6690.0,6720.0,27300000
3,2020-01-07,5581.578613,6740.0,6770.0,6730.0,6735.0,45022500
4,2020-01-08,5531.890137,6680.0,6745.0,6670.0,6670.0,53692500


## Step 3 - Feature Engineering

In [ ]:
df["Return"] = df["Close"].pct_change()
df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()
df["Change (%)"] = ((df["Close"]-df["Open"])/df["Open"])*100
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month_name()

display(df.head())


,Date,Adj Close,Close,High,Low,Open,Volume,Return,MA20,MA50,Change (%),Year,Month
0,2020-01-02,5540.172363,6690.0,6780.0,6680.0,6695.0,49445000,NaN,NaN,NaN,-0.074683,2020,January
1,2020-01-03,5631.266113,6800.0,6800.0,6725.0,6750.0,47755500,0.016442,NaN,NaN,0.740741,2020,January
2,2020-01-06,5577.437500,6735.0,6750.0,6690.0,6720.0,27300000,-0.009559,NaN,NaN,0.223214,2020,January
3,2020-01-07,5581.578613,6740.0,6770.0,6730.0,6735.0,45022500,0.000742,NaN,NaN,0.074239,2020,January
4,2020-01-08,5531.890137,6680.0,6745.0,6670.0,6670.0,53692500,-0.008902,NaN,NaN,0.149925,2020,January


## Step 4 - Simpan ke SQLite

In [ ]:
conn = sqlite3.connect("saham.db")
df.to_sql("harga_saham", conn, if_exists="replace", index=False)
print("Data berhasil disimpan ke SQLite")


Data berhasil disimpan ke SQLite


## Step 5 - Analisis SQL

In [ ]:
query = '''
SELECT *
FROM harga_saham
GROUP BY Year, Month
ORDER BY Year;
'''

hasil = pd.read_sql(query, conn)
display(hasil.head())


,Date,Adj Close,Close,High,Low,Open,Volume,Return,MA20,MA50,Change (%),Year,Month
0,2020-04-01 00:00:00,4538.138672,5480.0,5750.0,5425.0,5560.0,152831500,-0.008145,5452.0,6160.9,-1.438849,2020,April
1,2020-08-03 00:00:00,5164.178223,6130.0,6235.0,5825.0,6220.0,137079000,-0.017628,6143.0,5790.4,-1.446945,2020,August
2,2020-12-01 00:00:00,5387.425781,6395.0,6435.0,6220.0,6220.0,114563500,0.030620,6412.0,5981.1,2.813505,2020,December
3,2020-02-03 00:00:00,5333.139648,6440.0,6585.0,6370.0,6480.0,149797000,-0.006173,6764.5,NaN,-0.617284,2020,February
4,2020-01-02 00:00:00,5540.172363,6690.0,6780.0,6680.0,6695.0,49445000,NaN,NaN,NaN,-0.074683,2020,January


## Step 6 - Export ke Excel

In [ ]:
output_file = "dashboard_data.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Raw_Data", index=False)
    hasil.to_excel(writer, sheet_name="SQL_Result", index=False)

print(f"File berhasil dibuat: {output_file}")


File berhasil dibuat: dashboard_data.xlsx


## Step 7 - Dashboard Excel

Di Excel buatlah:
- KPI (Average Close, Highest, Lowest, Total Volume)
- Pivot Table
- Pivot Chart
- Slicer (Year, Month)
- Line Chart Harga
- Column Chart Volume
- Dashboard interaktif
